In [166]:
import numpy as np
import pandas as pd

# Parameters 

In [206]:
seed = 42
rng = np.random.default_rng(seed)

# variance components
A = .5
C = .2
E = 1.0 - A - C

sd_A = np.sqrt(A)
sd_C = np.sqrt(C)
sd_E = np.sqrt(E)

N = 1000000  # number of individuals (per generation)
ngen = 3     # number of generations to export

fam_size = 2.7
p_mztwin = 0.02
p_nonsocial_father = 0.15

In [207]:
# initalize
sex = rng.binomial(size=N, n=1, p=0.5)

a_pheno     = rng.normal(size=N, loc=0, scale=sd_A)
c_household = rng.normal(size=N, loc=0, scale=sd_C)
e           = rng.normal(size=N, loc=0, scale=sd_E)

pheno = np.stack([a_pheno, c_household, e], axis=-1)

In [208]:
def mating(
    parental_sex,
    fam_size,
    p_nonsocial_father,
    p_mztwin,

):
    # generate a pairing of parents that respects the inputs, ie:
    # one parent of each sex
    # mean size of families (excluding non-mating inds)
    # proportion of nonscocial fathers
    # proportion of mztwins

    # males have sex = 1
    n = len(parental_sex)
    nmale = parental_sex.sum()
    nfemale = n - nmale
    enough_offspring_check = True
    male_idxs = np.where(parental_sex)[0]
    female_idxs = np.where(parental_sex == 0)[0]

    rng.shuffle(female_idxs)
    rng.shuffle(male_idxs)
    
    while enough_offspring_check:
        family_sizes = rng.poisson(lam=fam_size, size=nfemale)
        if family_sizes.sum() >= n:
            enough_offspring_check = False

    # generate n offspring
    parent_idxs = np.zeros(shape = (n, 2), dtype = int)
    remaining_offspring = n
    n_nonsocial = 0
    twins = []
    for i in range(nfemale):
        mother = female_idxs[i]
        social_father = male_idxs[i]
        num_kids = family_sizes[i]

        if num_kids > remaining_offspring:
            num_kids = remaining_offspring
        
        for k in range(num_kids):
            if rng.uniform() > p_nonsocial_father:
                bio_father = social_father
            else:
                bio_father = rng.choice(male_idxs)
                n_nonsocial += 1

            if rng.uniform() > p_mztwin:
                # not a twin
                parent_idxs[remaining_offspring-1] = [mother, bio_father]
                remaining_offspring -= 1
                next
            else:
                # mz twins, unless only one spot remains in the gen
                parent_idxs[remaining_offspring-1] = [mother, bio_father]
                remaining_offspring -= 1
                if remaining_offspring > 0:
                    parent_idxs[remaining_offspring-1] = [mother, bio_father]
                    remaining_offspring -= 1
                    twins.append([remaining_offspring+1, remaining_offspring])
        
        if remaining_offspring == 0:
            break
    
    return parent_idxs, np.array(twins, dtype=int) 

In [209]:
def reproduce(pheno, parents, twins):
    n = len(pheno)
    mp = pheno[:,0][parents].mean(1)
    a_offspring = rng.normal(size=n, loc=mp, scale=sd_A/np.sqrt(2))
    c_offspring = pheno[:,1][parents[:,0]] # get from mothers
    e_offspring = rng.normal(size=n, loc=0, scale=sd_E)
    sex_offspring = rng.binomial(size=n, n=1, p=0.5)


    # set the additive genetic factor and sex of mz twins to be equal
    a_offspring[twins[:,1]] = a_offspring[twins[:,0]]
    sex_offspring[twins[:,1]] = sex_offspring[twins[:,0]]
    
    offspring = np.stack([a_offspring, c_offspring, e_offspring], axis=-1)
    
    return offspring, sex_offspring

In [210]:
def add_to_pedigree( pheno, sex, parents, twins, pedigree=None):
    df = pd.DataFrame(pheno)
    df.columns = ['A', 'C', 'E']
    df['sex'] = sex
    df[['mother', 'father']] = parents
    df['twin'] = -1
    df.loc[twins[:,0], 'twin'] = twins[:,1]
    df.loc[twins[:,1], 'twin'] = twins[:,0]
    df['id'] = df.index.values
    df = df[['id', 'sex', 'mother', 'father', 'twin', 'A', 'C', 'E']]

    if pedigree is not None:
        n = len(df)
        offset_id = pedigree['id'].max() + 1
        offset_parent = offset_id - n
        df['id'] = df['id'] + offset_id
        df['mother'] = df['mother'] + offset_parent
        df['father'] = df['father'] + offset_parent
        pedigree = pd.concat([pedigree, df]).reset_index(drop=True)
        
    else:
        # first gen, so set parents to unknown
        df['mother'] = -1
        df['father'] = -1
        pedigree = df
    return pedigree

In [211]:
NGEN = 3
pedigree = None
for i in range(NGEN):
    parents, twins = mating(parental_sex=sex, fam_size=fam_size, p_nonsocial_father=p_nonsocial_father, p_mztwin=p_mztwin)
    pheno, sex = reproduce(pheno, parents, twins)
    pedigree = add_to_pedigree(pheno, sex, parents, twins, pedigree=pedigree )


In [212]:
pedigree

,id,sex,mother,father,twin,A,C,E
0,0,0,-1,-1,-1,-0.325654,1.316343,0.085982
1,1,1,-1,-1,-1,-0.476593,0.894963,0.115308
2,2,0,-1,-1,-1,-0.306210,0.894963,0.906271
3,3,1,-1,-1,-1,-0.114981,0.894963,-0.552023
4,4,1,-1,-1,5,0.213899,0.894963,-0.394633
...,...,...,...,...,...,...,...,...
2999995,2999995,0,1895708,1700128,-1,-0.200195,0.409464,0.107204
2999996,2999996,1,1916792,1775473,-1,-1.039950,-0.876280,-0.525410
2999997,2999997,1,1916792,1528900,-1,-0.140308,-0.876280,0.687477
2999998,2999998,1,1916792,1528900,-1,0.752468,-0.876280,-0.709633


In [202]:
pedigree.iloc[9990:10002]

,id,sex,mother,father,twin,A,C,E
9990,9990,0,-1,-1,-1,0.456952,-0.748872,-0.094120
9991,9991,1,-1,-1,-1,-0.012248,-0.748872,-0.189334
9992,9992,1,-1,-1,-1,-1.032253,-0.373279,-0.276125
9993,9993,0,-1,-1,-1,-1.195045,-0.373279,-1.585663
9994,9994,1,-1,-1,-1,-0.939643,0.226954,0.135701
9995,9995,0,-1,-1,-1,0.720544,0.226954,0.755774
9996,9996,0,-1,-1,-1,-0.584489,-0.764088,-0.003133
9997,9997,0,-1,-1,-1,0.424159,-0.764088,-0.597075
9998,9998,1,-1,-1,-1,-0.048214,-0.764088,0.639380
9999,9999,0,-1,-1,-1,-0.212845,-0.764088,-0.109403


In [214]:
ped= add_to_pedigree(pheno, sex, parents, twins)

In [220]:
(ped.twin !=-1).sum()/1000000

np.float64(0.039362)

In [177]:
twins

array([[9948, 9947],
       [9938, 9937],
       [9876, 9875],
       [9856, 9855],
       [9760, 9759],
       [9757, 9756],
       [9742, 9741],
       [9674, 9673],
       [9533, 9532],
       [9341, 9340],
       [9294, 9293],
       [9243, 9242],
       [9172, 9171],
       [9159, 9158],
       [9147, 9146],
       [9110, 9109],
       [9061, 9060],
       [8998, 8997],
       [8974, 8973],
       [8895, 8894],
       [8892, 8891],
       [8859, 8858],
       [8801, 8800],
       [8767, 8766],
       [8727, 8726],
       [8693, 8692],
       [8643, 8642],
       [8579, 8578],
       [8530, 8529],
       [8369, 8368],
       [8163, 8162],
       [8154, 8153],
       [8140, 8139],
       [8119, 8118],
       [8086, 8085],
       [8065, 8064],
       [8011, 8010],
       [8004, 8003],
       [7934, 7933],
       [7925, 7924],
       [7902, 7901],
       [7716, 7715],
       [7689, 7688],
       [7677, 7676],
       [7628, 7627],
       [7560, 7559],
       [7551, 7550],
       [7510,

In [155]:
# check
for x in parents[twins]:
    assert x[0, 0] == x[1,0]
    assert x[0, 1] == x[1,1]